# Lecture 2 · Linear Models for Classification II — Exercises

Companion to the note [Lecture Notes 1-12](../Lecture%20Notes%201-12.md).

Practise the material of **Lecture 2**: generative vs. discriminative classifiers, Naive Bayes and Gaussian class-conditionals,
**Linear Discriminant Analysis** (shared covariance, MLE estimates, the linear boundary $\mathbf{w} = \boldsymbol\Sigma^{-1}(\boldsymbol\mu_k-\boldsymbol\mu_j)$),
**logistic regression** (sigmoid, log-odds, cross-entropy and its gradient $\sum_n(\sigma_n - y_n)\mathbf{x}_n$), the maximum-margin
**SVM** with its Lagrangian dual and support vectors, and non-linear feature maps $\boldsymbol\phi(\mathbf{x})$ (XOR). Code exercises
rebuild LDA, Gaussian Naive Bayes and logistic regression in NumPy and compare them with the scikit-learn models used in `MLSelfStudy2-F26.ipynb`.

**20 exercises** · 🧠 Concept ×5 · ✍️ By hand ×8 · 💻 Code ×7

**How to use this notebook**
- Difficulty: ★☆☆ warm-up · ★★☆ standard · ★★★ challenge.
- ✍️ *By hand* exercises are meant for pen and paper (or a markdown cell). 💻 *Code* exercises have a cell for you to fill in.
- Every exercise has a folded **💡 Hint** and **✅ Solution**. Click to unfold, but try first!
- The notebook runs top to bottom as-is; `check(...)` lines print ⏳ until you have an answer, then ✅ or ❌.


In [ ]:
import numpy as np
np.random.seed(0)

# --- helper: compares your answer with a reference, prints ✅/❌, never raises ---
import numpy as _np
def check(name, got, expected, tol=1e-6):
    if got is None or got is Ellipsis:
        print(f"⏳ {name}: not attempted yet")
        return
    try:
        ok = bool(_np.allclose(_np.asarray(got, dtype=float), _np.asarray(expected, dtype=float), atol=tol, rtol=1e-4))
    except Exception:
        try:
            ok = bool(got == expected)
        except Exception:
            ok = False
    print(("✅ " if ok else "❌ ") + name + ("" if ok else f"  (got {got!r})"))

import matplotlib.pyplot as plt
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.datasets import make_classification, make_circles
from scipy.special import expit
rng = np.random.default_rng(2)

## Part A · Concepts

### Exercise 1 · Generative or discriminative?
*🧠 Concept · ★☆☆*

For each model say whether it is generative or discriminative (or neither: non-probabilistic), and which of these tasks it can do:
(i) predict $Y$ from $\mathbf{x}$, (ii) fill in a missing feature $X_i$, (iii) sample new synthetic pairs $(\mathbf{x}, y)$.

Naive Bayes · LDA · Logistic regression · Least-squares classification · Perceptron · Linear SVM

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Only a model of the joint $P(\mathbf{X}, Y)$ can answer questions about $\mathbf{X}$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

| Model | Paradigm | (i) | (ii) | (iii) |
|---|---|---|---|---|
| Naive Bayes | generative ($P(Y)\prod_i P(X_i\mid Y)$) | ✔ | ✔ | ✔ |
| LDA | generative (Gaussians, shared $\boldsymbol\Sigma$) | ✔ | ✔ | ✔ |
| Logistic regression | discriminative ($P(Y\mid\mathbf{X})$) | ✔ | ✘ | ✘ |
| Least squares / Perceptron / SVM | non-probabilistic | ✔ (labels only) | ✘ | ✘ |

Generative models pay for (ii) and (iii) by having to model $P(\mathbf{X}\mid Y)$, which is wasted effort (and a source of bias) if all you need is (i).

</details>

### Exercise 2 · Naive Bayes, GMM, LDA, QDA: which assumption gives which boundary?
*🧠 Concept · ★★☆*

All four model $P(\mathbf{x}\mid Y=k)$ as Gaussian. Fill in for each: the form of $\boldsymbol\Sigma_k$, and whether the decision boundary is linear or quadratic.

(a) Gaussian Naive Bayes, (b) full Gaussian per class ("GMM classifier", a.k.a. QDA), (c) LDA.

Explain in one sentence *why* LDA's boundary is linear, referring to the log-posterior ratio.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Expand $-\tfrac12(\mathbf{x}-\boldsymbol\mu_k)^T\boldsymbol\Sigma_k^{-1}(\mathbf{x}-\boldsymbol\mu_k)$. Which term is quadratic in $\mathbf{x}$, and when does it cancel between two classes?

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) Naive Bayes: $\boldsymbol\Sigma_k = \operatorname{diag}(\sigma_{1,k}^2,\dots,\sigma_{D,k}^2)$, different per class, so the boundary is **quadratic** in general (linear only if the variances are shared).
(b) Full Gaussian per class: unrestricted $\boldsymbol\Sigma_k$, so the boundary is **quadratic**.
(c) LDA: one shared $\boldsymbol\Sigma$, so the boundary is **linear**.

In $\ln\frac{P(k\mid\mathbf{x})}{P(j\mid\mathbf{x})}$ the term $-\tfrac12\mathbf{x}^T\boldsymbol\Sigma^{-1}\mathbf{x}$ appears in both classes with the same $\boldsymbol\Sigma$ and cancels,
leaving $\mathbf{x}^T\boldsymbol\Sigma^{-1}(\boldsymbol\mu_k-\boldsymbol\mu_j) + \text{const}$, which is linear (the $\ln|\boldsymbol\Sigma|$ terms cancel too).

</details>

### Exercise 3 · Logistic regression on separable data
*🧠 Concept · ★★☆*

The comparison table says that on linearly separable data logistic regression "converges asymptotically; weights diverge to infinity without
regularization". Explain why, using the cross-entropy $E(\mathbf{w}, b)$. What does scikit-learn's default `LogisticRegression()` do to prevent it?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

If $\mathbf{w}$ separates the data, what happens to $E(c\,\mathbf{w}, c\,b)$ as $c\to\infty$?

</details>

<details>
<summary><b>✅ Solution</b></summary>

If $(\mathbf{w}, b)$ separates the data, every $y_n=1$ point has $\mathbf{w}^T\mathbf{x}_n+b>0$ and every $y_n=0$ point has it $<0$. Scaling by $c>1$ pushes all
$\sigma_n$ closer to their targets, so $E(c\mathbf{w}, cb)$ decreases strictly and tends to 0 as $c\to\infty$; it is never attained, the infimum is at infinity.
Gradient-based solvers keep increasing $\lVert\mathbf{w}\rVert$ (probabilities become 0/1, overconfident). scikit-learn adds an L2 penalty by default
(`penalty='l2', C=1.0`), i.e. it minimises $E + \frac{1}{2C}\lVert\mathbf{w}\rVert^2$, which has a finite minimiser. (In [[Lecture Notes 1-12#Lecture 6: Probabilistic Models and Learning]]
this penalty reappears as a Gaussian prior.)

</details>

### Exercise 4 · Who needs feature scaling?
*🧠 Concept · ★☆☆*

The self-study standardises features with `StandardScaler` before fitting LDA, logistic regression and a linear SVC.
For each model explain how much scaling matters, and why `scaler.transform` (not `fit_transform`) is used on the test set.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Which of the models measure distances/norms in feature space directly, and which estimate a full covariance?

</details>

<details>
<summary><b>✅ Solution</b></summary>

* **Linear SVC**: matters a lot. The margin is measured with $\lVert\mathbf{w}\rVert$; features with large scale dominate the geometry.
* **Logistic regression**: the unregularised optimum is scale-equivariant, but optimisation (conditioning of the gradient) and the L2 penalty are not, so scaling helps convergence and changes the regularised solution.
* **LDA**: predictions are invariant to (invertible affine) rescaling since $\boldsymbol\Sigma$ absorbs the scales; scaling only helps numerically.

The test set must be transformed with the **training** mean/std; fitting the scaler on test data leaks test information and makes train and test features inconsistent.

</details>

### Exercise 5 · Only the support vectors matter
*🧠 Concept · ★★☆*

Using the KKT complementary slackness condition $\lambda_n\{y_n(\mathbf{w}^T\mathbf{x}_n + b) - 1\} = 0$, explain why moving a training point that is
strictly outside the margin (without crossing it) leaves the SVM solution unchanged, while the same is **not** true for LDA or logistic regression.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

$\mathbf{w} = \sum_n\lambda_n y_n\mathbf{x}_n$. What is $\lambda_n$ for a point with $y_n(\mathbf{w}^T\mathbf{x}_n+b)>1$?

</details>

<details>
<summary><b>✅ Solution</b></summary>

For a point strictly outside the margin, the bracket is $>0$, so complementary slackness forces $\lambda_n = 0$; it does not appear in
$\mathbf{w} = \sum_n\lambda_n y_n\mathbf{x}_n$ nor in the active constraints. Moving it (while keeping it outside the margin) keeps the same KKT point optimal.
LDA uses **all** points through the class means and the pooled covariance, and logistic regression gives every point a non-zero gradient contribution
$(\sigma_n - y_n)\mathbf{x}_n$, so both move. Exercise 20 verifies this numerically.

</details>

## Part B · By hand

### Exercise 6 · Spam posterior with Bayes' theorem
*✍️ By hand · ★☆☆*

A generative spam filter has $P(Y=1) = 0.2$, $P(X=1\mid Y=1) = 0.7$, $P(X=1\mid Y=0) = 0.05$, where $X$ = "contains the word *Winner*".
Compute $P(Y=1\mid X=1)$ and $P(Y=1\mid X=0)$.

In [ ]:
p_spam_given_winner = None
p_spam_given_no_winner = None

_py = np.array([0.8, 0.2]); _px1 = np.array([0.05, 0.7])
_j1 = _py * _px1; _j0 = _py * (1 - _px1)
check('P(Y=1|X=1)', p_spam_given_winner, _j1[1] / _j1.sum(), tol=1e-3)
check('P(Y=1|X=0)', p_spam_given_no_winner, _j0[1] / _j0.sum(), tol=1e-3)

<details>
<summary><b>💡 Hint</b></summary>

Joint: $P(X=1, Y=1) = 0.7\cdot0.2$; $P(X=1,Y=0) = 0.05\cdot0.8$. For $X=0$ use $1-0.7$ and $1-0.05$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$P(Y=1\mid X=1) = \frac{0.14}{0.14 + 0.04} = \frac{0.14}{0.18} \approx \mathbf{0.778}$.

$P(Y=1\mid X=0) = \frac{0.3\cdot0.2}{0.3\cdot0.2 + 0.95\cdot0.8} = \frac{0.06}{0.82} \approx \mathbf{0.0732}$.

```python
p_spam_given_winner = 0.14 / 0.18
p_spam_given_no_winner = 0.06 / 0.82
```

</details>

### Exercise 7 · Logistic regression probabilities
*✍️ By hand · ★☆☆*

A trained model has $w_0 = -4.5$, $\mathbf{w} = (1.5, 0.5)^T$. Compute $P(\mathcal{C}_1\mid\mathbf{x})$ for $\mathbf{x} = (2, 3)^T$ and $\mathbf{x}' = (4, 3)^T$,
and the **odds** $P(\mathcal C_1\mid\mathbf{x}')/P(\mathcal C_0\mid\mathbf{x}')$. By what factor do the odds change when $x_1$ increases by 1?

In [ ]:
p_x = None
p_xprime = None
odds_xprime = None
odds_factor_per_x1 = None

_w = np.array([-4.5, 1.5, 0.5])
_a, _b = _w @ [1, 2, 3], _w @ [1, 4, 3]
check('P(C1|x)', p_x, expit(_a), tol=1e-3)
check("P(C1|x')", p_xprime, expit(_b), tol=1e-3)
check("odds(x')", odds_xprime, expit(_b) / (1 - expit(_b)), tol=1e-2)
check('odds factor', odds_factor_per_x1, (expit(_a + 1.5) / (1 - expit(_a + 1.5))) / (expit(_a) / (1 - expit(_a))), tol=1e-3)

<details>
<summary><b>💡 Hint</b></summary>

The log-odds equal the activation $a = \mathbf{w}^T\mathbf{x} + w_0$, so the odds are $e^a$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$a = -4.5 + 3 + 1.5 = 0 \Rightarrow \sigma(0) = \mathbf{0.5}$ (on the boundary).
$a' = -4.5 + 6 + 1.5 = 3 \Rightarrow \sigma(3) = 1/(1+e^{-3}) \approx \mathbf{0.9526}$, odds $= e^{3}\approx\mathbf{20.09}$.
Increasing $x_1$ by 1 adds $w_1 = 1.5$ to the log-odds, i.e. multiplies the odds by $e^{1.5}\approx\mathbf{4.48}$, regardless of $\mathbf{x}$.

```python
p_x = 0.5
p_xprime = 1 / (1 + np.exp(-3))
odds_xprime = np.exp(3)
odds_factor_per_x1 = np.exp(1.5)
```

</details>

### Exercise 8 · Sigmoid derivative and the cross-entropy gradient
*✍️ By hand · ★★☆*

(a) Show $\sigma'(z) = \sigma(z)(1-\sigma(z))$ and $1-\sigma(z) = \sigma(-z)$.
(b) Show that $\ln\frac{\sigma(z)}{1-\sigma(z)} = z$.
(c) Derive $\nabla_\mathbf{w}E = \sum_n(\sigma_n - y_n)\mathbf{x}_n$ for the cross-entropy error, with $\sigma_n = \sigma(\mathbf{w}^T\mathbf{x}_n + b)$.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

For (c), differentiate one term: $\frac{\partial}{\partial z}\left[-y\ln\sigma(z) - (1-y)\ln(1-\sigma(z))\right]$ and simplify, then apply $\partial z/\partial\mathbf{w} = \mathbf{x}_n$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) $\sigma(z) = (1+e^{-z})^{-1}$, so $\sigma'(z) = \frac{e^{-z}}{(1+e^{-z})^2} = \frac{1}{1+e^{-z}}\cdot\frac{e^{-z}}{1+e^{-z}} = \sigma(z)(1-\sigma(z))$.
Also $1-\sigma(z) = \frac{e^{-z}}{1+e^{-z}} = \frac{1}{e^{z}+1} = \sigma(-z)$.

(b) $\frac{\sigma(z)}{1-\sigma(z)} = \frac{1}{e^{-z}} = e^z$, so the log-odds equal $z$.

(c) $\frac{\partial}{\partial z}[-y\ln\sigma - (1-y)\ln(1-\sigma)] = -y\frac{\sigma(1-\sigma)}{\sigma} + (1-y)\frac{\sigma(1-\sigma)}{1-\sigma} = -y(1-\sigma) + (1-y)\sigma = \sigma - y$.
With the chain rule $\partial z_n/\partial\mathbf{w} = \mathbf{x}_n$: $\nabla_\mathbf{w}E = \sum_n(\sigma_n-y_n)\mathbf{x}_n$ and $\partial E/\partial b = \sum_n(\sigma_n - y_n)$.

</details>

### Exercise 9 · LDA in one dimension
*✍️ By hand · ★★☆*

Class 0: $x\in\{1, 2, 3\}$, class 1: $x\in\{5, 6, 7\}$. Compute the MLE priors, means, the **shared** variance
$\sigma^2 = \frac1N\sum_k\sum_{n:y_n=k}(x_n-\mu_k)^2$, and the LDA discriminant $w x + w_0$ for "class 1 vs class 0"
($w = \sigma^{-2}(\mu_1-\mu_0)$, $w_0 = -\tfrac12\mu_1^2/\sigma^2 + \tfrac12\mu_0^2/\sigma^2 + \ln\frac{P(1)}{P(0)}$). Where is the boundary?

In [ ]:
w_lda1d = None
w0_lda1d = None
boundary_lda1d = None

_m = LinearDiscriminantAnalysis().fit(np.array([1, 2, 3, 5, 6, 7.])[:, None], np.array([0, 0, 0, 1, 1, 1]))
check('w', w_lda1d, _m.coef_[0, 0]); check('w0', w0_lda1d, _m.intercept_[0])
check('boundary', boundary_lda1d, -_m.intercept_[0] / _m.coef_[0, 0])

<details>
<summary><b>💡 Hint</b></summary>

Means 2 and 6; each class has squared deviations $1+0+1 = 2$, so $\sigma^2 = 4/6$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Priors $\tfrac12,\tfrac12$; $\mu_0 = 2$, $\mu_1 = 6$; $\sigma^2 = (2+2)/6 = \tfrac23$.
$w = (6-2)/\tfrac23 = \mathbf{6}$, $w_0 = \tfrac32(-18 + 2) + \ln 1 = \mathbf{-24}$. Boundary: $6x - 24 = 0\Rightarrow x = \mathbf{4}$, halfway between the means
(equal priors). scikit-learn's `coef_`/`intercept_` are exactly these MLE quantities.

```python
w_lda1d = 6.0
w0_lda1d = -24.0
boundary_lda1d = 4.0
```

</details>

### Exercise 10 · Counting parameters
*✍️ By hand · ★★☆*

$D = 10$ continuous features, $K = 3$ classes. Count the free parameters of
(a) Gaussian Naive Bayes, (b) a full Gaussian per class (QDA/"GMM classifier"), (c) LDA, (d) multinomial logistic regression with a reference class.
(Priors: $K-1$ free numbers; a symmetric $D\times D$ covariance has $D(D+1)/2$.)

In [ ]:
n_nb = None
n_qda = None
n_lda = None
n_logreg = None

D, K = 10, 3
check('Naive Bayes', n_nb, K*D + K*D + (K-1))
check('QDA', n_qda, K*D + K*D*(D+1)//2 + (K-1))
check('LDA', n_lda, K*D + D*(D+1)//2 + (K-1))
check('logistic regression', n_logreg, (K-1)*(D+1))

<details>
<summary><b>💡 Hint</b></summary>

Means: $KD$ in every generative model. The difference is in the covariances.

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) $30 + 30 + 2 = \mathbf{62}$. (b) $30 + 3\cdot55 + 2 = \mathbf{197}$. (c) $30 + 55 + 2 = \mathbf{87}$. (d) $(K-1)(D+1) = \mathbf{22}$.
The discriminative model needs far fewer parameters for the same (linear) decision rule as LDA: the "data efficiency" row of the comparison table.

```python
n_nb, n_qda, n_lda, n_logreg = 62, 197, 87, 22
```

</details>

### Exercise 11 · A maximum-margin hyperplane by hand
*✍️ By hand · ★★☆*

Training set: $\mathbf{x}_1 = (1,1)$, $y_1 = +1$; $\mathbf{x}_2 = (-1,-1)$, $y_2 = -1$; $\mathbf{x}_3 = (3, 3)$, $y_3 = +1$.

(a) Argue which points are support vectors and find $\mathbf{w}, b$ of the maximum-margin hyperplane (constraints $y_n(\mathbf{w}^T\mathbf{x}_n+b)\ge1$ tight at the SVs).
(b) Compute the margin $1/\lVert\mathbf{w}\rVert$.
(c) Find the Lagrange multipliers from $\mathbf{w} = \sum_n\lambda_n y_n\mathbf{x}_n$ and $\sum_n\lambda_n y_n = 0$.

In [ ]:
w_svm = None      # [w1, w2]
b_svm = None
margin_svm = None
lambdas_svm = None  # [l1, l2, l3]

_X = np.array([[1, 1], [-1, -1], [3, 3.]]); _y = np.array([1, -1, 1])
_s = SVC(kernel='linear', C=1e6).fit(_X, _y)
_lam = np.zeros(3); _lam[_s.support_] = np.abs(_s.dual_coef_[0])
check('w', w_svm, _s.coef_[0], tol=1e-3); check('b', b_svm, _s.intercept_[0], tol=1e-3)
check('margin', margin_svm, 1 / np.linalg.norm(_s.coef_[0]), tol=1e-3)
check('lambdas', lambdas_svm, _lam, tol=1e-3)

<details>
<summary><b>💡 Hint 1</b></summary>

$\mathbf{x}_3$ lies behind $\mathbf{x}_1$, so it cannot be closest to the boundary.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

By symmetry $\mathbf{w} = (a, a)$ and $b = 0$; impose $\mathbf{w}^T\mathbf{x}_1 = 1$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) SVs: $\mathbf{x}_1, \mathbf{x}_2$. The perpendicular bisector of $\mathbf{x}_1\mathbf{x}_2$ is $x_1 + x_2 = 0$; with $\mathbf{w} = (a,a)$, $b=0$ and $2a = 1$: $\mathbf{w} = (0.5, 0.5)$, $b = 0$.
Check $\mathbf{x}_3$: $y_3\,\mathbf{w}^T\mathbf{x}_3 = 3 > 1$, outside the margin.

(b) $\lVert\mathbf{w}\rVert = 1/\sqrt2$, so the margin is $\sqrt2\approx\mathbf{1.414}$, exactly the distance from each SV to the boundary.

(c) $\lambda_3 = 0$; $\lambda_1 = \lambda_2 = \lambda$ from $\sum\lambda_n y_n = 0$; $\mathbf{w} = \lambda(1,1) - \lambda(-1,-1) = 2\lambda(1,1) = (0.5,0.5)\Rightarrow\lambda = \mathbf{0.25}$.

```python
w_svm = [0.5, 0.5]
b_svm = 0.0
margin_svm = np.sqrt(2)
lambdas_svm = [0.25, 0.25, 0.0]
```

</details>

### Exercise 12 · From the Lagrangian to the Wolfe dual
*✍️ By hand · ★★★*

Starting from $L(\mathbf{w}, b, \boldsymbol\lambda) = \frac12\lVert\mathbf{w}\rVert^2 - \sum_n\lambda_n\{y_n(\mathbf{w}^T\mathbf{x}_n+b)-1\}$, set the derivatives w.r.t. $\mathbf{w}$ and $b$ to
zero and substitute back to obtain the dual objective
$\sum_n\lambda_n - \frac12\sum_n\sum_m\lambda_n\lambda_m y_n y_m\,\mathbf{x}_n^T\mathbf{x}_m$. Why is it important that the data only enter through $\mathbf{x}_n^T\mathbf{x}_m$?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Expand $L$: the $b$ term is $-b\sum_n\lambda_n y_n = 0$. Then $\mathbf{w}^T\sum_n\lambda_ny_n\mathbf{x}_n = \lVert\mathbf{w}\rVert^2$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$\partial L/\partial\mathbf{w} = 0 \Rightarrow \mathbf{w} = \sum_n\lambda_ny_n\mathbf{x}_n$; $\partial L/\partial b = 0\Rightarrow\sum_n\lambda_ny_n = 0$.
Expand: $L = \frac12\lVert\mathbf{w}\rVert^2 - \mathbf{w}^T\sum_n\lambda_ny_n\mathbf{x}_n - b\sum_n\lambda_ny_n + \sum_n\lambda_n = \frac12\lVert\mathbf{w}\rVert^2 - \lVert\mathbf{w}\rVert^2 - 0 + \sum_n\lambda_n$
$= \sum_n\lambda_n - \frac12\left\lVert\sum_n\lambda_ny_n\mathbf{x}_n\right\rVert^2 = \sum_n\lambda_n - \frac12\sum_{n,m}\lambda_n\lambda_my_ny_m\,\mathbf{x}_n^T\mathbf{x}_m$,
maximised subject to $\lambda_n\ge0$, $\sum_n\lambda_ny_n = 0$.

Because only inner products appear (also in the prediction $\sum_{n\in SV}\lambda_ny_n\mathbf{x}_n^T\mathbf{z} + b$), we can replace them by a kernel
$K(\mathbf{x}_n, \mathbf{x}_m) = \boldsymbol\phi(\mathbf{x}_n)^T\boldsymbol\phi(\mathbf{x}_m)$ and work in huge feature spaces without ever computing $\boldsymbol\phi$:
the kernel trick of [[Lecture Notes 1-12#Lecture 3: Kernel Support Vector Machines]].

</details>

### Exercise 13 · XOR becomes linear in feature space
*✍️ By hand · ★★☆*

(a) Prove that no line $w_1A + w_2B + w_0 = 0$ separates XOR (positive: $(1,0), (0,1)$; negative: $(0,0), (1,1)$).
(b) With $\boldsymbol\phi(A, B) = (A, B, AB)$, $\mathbf{w} = (1, 1, -2)$, $b = -0.5$, compute the four scores $\mathbf{w}^T\boldsymbol\phi + b$ in the order
$(1,1), (1,0), (0,1), (0,0)$.

In [ ]:
xor_scores = None  # 4 numbers

_P = np.array([[1, 1], [1, 0], [0, 1], [0, 0.]])
_Phi = np.c_[_P, _P[:, 0] * _P[:, 1]]
check('scores', xor_scores, _Phi @ np.array([1, 1, -2.]) - 0.5)

<details>
<summary><b>💡 Hint</b></summary>

Write the four inequalities and add the two positive ones and the two negative ones.

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) Need $w_1 + w_0 > 0$, $w_2 + w_0 > 0$ (positives) and $w_0 \le 0$, $w_1 + w_2 + w_0 \le 0$ (negatives). Adding the first two: $w_1 + w_2 + 2w_0 > 0$.
Adding the last two: $w_1 + w_2 + 2w_0 \le 0$. Contradiction.

(b) $(1,1)\mapsto 1+1-2-0.5 = -0.5$; $(1,0)\mapsto 0.5$; $(0,1)\mapsto 0.5$; $(0,0)\mapsto -0.5$. Perfect separation with a plane in $\mathbb{R}^3$, which is a
non-linear (hyperbolic) curve $A + B - 2AB = 0.5$ in the original space.

```python
xor_scores = [-0.5, 0.5, 0.5, -0.5]
```

</details>

## Part C · Code from scratch

### Exercise 14 · Cross-entropy and its gradient, with a gradient check
*💻 Code · ★☆☆*

Implement `cross_entropy(w, X, y)` (with the bias as `w[0]` and a column of ones prepended inside) and `ce_grad(w, X, y)` using the formula from Exercise 8.
Verify your gradient with central finite differences: store the max absolute difference in `grad_err`.
Use `np.logaddexp` for a numerically safe loss: $-\ln\sigma(z) = \ln(1+e^{-z})$.

In [ ]:
X_ce, y_ce = make_classification(n_samples=50, n_features=3, n_informative=3, n_redundant=0, random_state=0)
w_ce = rng.normal(size=4)

def cross_entropy(w, X, y):
    # TODO
    return None

def ce_grad(w, X, y):
    # TODO
    return None

grad_err = None

from sklearn.metrics import log_loss
_Xt = np.c_[np.ones(50), X_ce]
check('loss matches sklearn log_loss * N', cross_entropy(w_ce, X_ce, y_ce), log_loss(y_ce, expit(_Xt @ w_ce)) * 50)
check('gradient check < 1e-5', None if grad_err is None else grad_err < 1e-5, True)

<details>
<summary><b>💡 Hint</b></summary>

Finite difference: `(f(w + h e_i) - f(w - h e_i)) / (2h)` with `h = 1e-5`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$-[y\ln\sigma(z) + (1-y)\ln(1-\sigma(z))] = y\ln(1+e^{-z}) + (1-y)\ln(1+e^{z})$, which `np.logaddexp(0, -z)` evaluates without overflow.
A tiny finite-difference error confirms the analytic gradient.

```python
X_ce, y_ce = make_classification(n_samples=50, n_features=3, n_informative=3, n_redundant=0, random_state=0)
w_ce = rng.normal(size=4)

def cross_entropy(w, X, y):
    z = np.c_[np.ones(len(X)), X] @ w
    return np.sum(y * np.logaddexp(0, -z) + (1 - y) * np.logaddexp(0, z))

def ce_grad(w, X, y):
    Xt = np.c_[np.ones(len(X)), X]
    return Xt.T @ (expit(Xt @ w) - y)

h = 1e-5
num = np.array([(cross_entropy(w_ce + h * e, X_ce, y_ce) - cross_entropy(w_ce - h * e, X_ce, y_ce)) / (2 * h) for e in np.eye(4)])
grad_err = np.max(np.abs(num - ce_grad(w_ce, X_ce, y_ce)))
print('max abs difference:', grad_err)
```

</details>

### Exercise 15 · Logistic regression by gradient descent
*💻 Code · ★★☆*

Train logistic regression with batch gradient descent on the averaged gradient $\frac1N\sum_n(\sigma_n-y_n)\tilde{\mathbf{x}}_n$
(learning rate 0.5, 5000 iterations, start at zero) on `X_lr, y_lr`. Store $\tilde{\mathbf{w}} = (b, \mathbf{w})$ in `w_lr`.
Compare with `LogisticRegression(C=1e6)` (almost no regularisation) from scikit-learn (`intercept_`, `coef_`).

In [ ]:
X_lr, y_lr = make_classification(n_samples=300, n_features=3, n_informative=3, n_redundant=0, class_sep=0.8, random_state=0)

def fit_logreg_gd(X, y, lr=0.5, iters=5000):
    # TODO
    return None

w_lr = fit_logreg_gd(X_lr, y_lr)

_ref = LogisticRegression(C=1e6, tol=1e-10, max_iter=10000).fit(X_lr, y_lr)
check('GD = sklearn (unregularised)', w_lr, np.r_[_ref.intercept_, _ref.coef_[0]], tol=1e-3)

<details>
<summary><b>💡 Hint</b></summary>

Reuse `ce_grad` from the previous exercise, divided by N.

</details>

<details>
<summary><b>✅ Solution</b></summary>

The cross-entropy is convex, so GD with a small enough step reaches the same optimum as scikit-learn's L-BFGS (quasi-Newton, which needs far fewer iterations).
There is no closed form because $\sigma$ makes the stationarity equation $\sum_n(\sigma_n-y_n)\mathbf{x}_n = 0$ non-linear in $\mathbf{w}$.

```python
X_lr, y_lr = make_classification(n_samples=300, n_features=3, n_informative=3, n_redundant=0, class_sep=0.8, random_state=0)

def fit_logreg_gd(X, y, lr=0.5, iters=5000):
    Xt = np.c_[np.ones(len(X)), X]
    w = np.zeros(Xt.shape[1])
    for _ in range(iters):
        w -= lr * Xt.T @ (expit(Xt @ w) - y) / len(y)
    return w

w_lr = fit_logreg_gd(X_lr, y_lr)
print(w_lr.round(4))
```

</details>

### Exercise 16 · LDA from scratch
*💻 Code · ★★☆*

Implement `fit_lda(X, y)` for two classes returning `(w, w0, mu, Sigma)` with the MLE formulas from the note (priors $N_k/N$, class means,
pooled covariance with $1/N$) and the discriminant $\mathbf{w} = \boldsymbol\Sigma^{-1}(\boldsymbol\mu_1-\boldsymbol\mu_0)$, $w_0$ as in the note.
Fit it to the unbalanced data below and compare with `LinearDiscriminantAnalysis()` (`coef_`, `intercept_`, `means_`).

In [ ]:
r = np.random.default_rng(5)
C = [[1, .6], [.6, 1]]
X_lda = np.r_[r.multivariate_normal([0, 0], C, 60), r.multivariate_normal([2, 1], C, 140)]
y_lda = np.r_[np.zeros(60, int), np.ones(140, int)]

def fit_lda(X, y):
    # TODO
    return None, None, None, None

w_lda, w0_lda, mu_lda, Sigma_lda = fit_lda(X_lda, y_lda)

_m = LinearDiscriminantAnalysis().fit(X_lda, y_lda)
check('w', w_lda, _m.coef_[0]); check('w0 (includes ln prior ratio)', w0_lda, _m.intercept_[0]); check('means', mu_lda, _m.means_)

<details>
<summary><b>💡 Hint</b></summary>

`S = sum((X[y==k]-mu[k]).T @ (X[y==k]-mu[k]) for k in (0,1)) / len(X)`; do not forget $\ln(N_1/N_0)$ in $w_0$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

LDA is "one-step": no iterations, only empirical statistics and one matrix inverse. With unequal priors ($0.3$ vs $0.7$) the term $\ln\frac{0.7}{0.3}$ shifts
the boundary towards the minority class mean.

```python
r = np.random.default_rng(5)
C = [[1, .6], [.6, 1]]
X_lda = np.r_[r.multivariate_normal([0, 0], C, 60), r.multivariate_normal([2, 1], C, 140)]
y_lda = np.r_[np.zeros(60, int), np.ones(140, int)]

def fit_lda(X, y):
    prior = np.array([np.mean(y == k) for k in (0, 1)])
    mu = np.array([X[y == k].mean(0) for k in (0, 1)])
    S = sum((X[y == k] - mu[k]).T @ (X[y == k] - mu[k]) for k in (0, 1)) / len(X)
    Si = np.linalg.inv(S)
    w = Si @ (mu[1] - mu[0])
    w0 = -0.5 * mu[1] @ Si @ mu[1] + 0.5 * mu[0] @ Si @ mu[0] + np.log(prior[1] / prior[0])
    return w, w0, mu, S

w_lda, w0_lda, mu_lda, Sigma_lda = fit_lda(X_lda, y_lda)
```

</details>

### Exercise 17 · Gaussian Naive Bayes: classify and generate
*💻 Code · ★★☆*

Implement Gaussian Naive Bayes on Iris (all 4 features): estimate priors, per-class means and per-class **diagonal** variances (MLE, $1/N_k$),
and compute the posterior `post_nb` (shape $150\times3$) via Bayes' theorem in log-space. Compare with `GaussianNB().predict_proba`.

Then use the fact that the model is **generative**: sample 5 new synthetic flowers of class 2 into `samples_c2`.

In [ ]:
from sklearn.datasets import load_iris
X_ir, y_ir = load_iris(return_X_y=True)
post_nb = None
samples_c2 = None

check('posterior = GaussianNB', post_nb, GaussianNB().fit(X_ir, y_ir).predict_proba(X_ir), tol=1e-4)
check('5 samples of dimension 4', None if samples_c2 is None else np.shape(samples_c2), (5, 4))

<details>
<summary><b>💡 Hint 1</b></summary>

$\ln P(\mathbf{x}\mid k) = \sum_i\left[-\tfrac12\ln(2\pi\sigma_{ik}^2) - \frac{(x_i-\mu_{ik})^2}{2\sigma_{ik}^2}\right]$.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

Normalise with log-sum-exp: subtract the row max before exponentiating.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Sampling is easy because the generative model is fully specified: draw $x_i\sim\mathcal N(\mu_{i,2}, \sigma^2_{i,2})$ independently. A logistic regression
could not do this. (`GaussianNB` adds a tiny `var_smoothing` to the variances, hence the tolerance.)

```python
from sklearn.datasets import load_iris
X_ir, y_ir = load_iris(return_X_y=True)
K = 3
prior = np.array([np.mean(y_ir == k) for k in range(K)])
mu = np.array([X_ir[y_ir == k].mean(0) for k in range(K)])
var = np.array([X_ir[y_ir == k].var(0) for k in range(K)])
loglik = np.stack([np.sum(-0.5 * np.log(2 * np.pi * var[k]) - (X_ir - mu[k]) ** 2 / (2 * var[k]), 1) for k in range(K)], 1)
logjoint = loglik + np.log(prior)
logjoint -= logjoint.max(1, keepdims=True)
post_nb = np.exp(logjoint) / np.exp(logjoint).sum(1, keepdims=True)
samples_c2 = rng.normal(mu[2], np.sqrt(var[2]), size=(5, 4))
print(samples_c2.round(2))
```

</details>

### Exercise 18 · When the shared-covariance assumption is wrong
*💻 Code · ★★☆*

Recreate Example 2 of [[Examples]]: class 0 $\sim\mathcal N((0,0), 0.1 I)$, class 1 $\sim\mathcal N((4,0), 9 I)$, 200 points each (use `rng`).
Fit `LinearDiscriminantAnalysis` and `QuadraticDiscriminantAnalysis` and store their training accuracies. Explain the gap.

In [ ]:
acc_lda_ex2 = None
acc_qda_ex2 = None

if acc_lda_ex2 is not None:
    check('QDA clearly beats LDA', acc_qda_ex2 > acc_lda_ex2 + 0.05, True)
else:
    check('QDA clearly beats LDA', None, True)

<details>
<summary><b>💡 Hint</b></summary>

`rng.normal(mean, std, size=(200, 2))` with std $=\sqrt{0.1}$ and $3$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

LDA pools the covariances into roughly $4.55I$ and puts a straight boundary near $x_1 = 2$; many wide class-1 points fall left of it (accuracy about 0.89).
QDA keeps $\boldsymbol\Sigma_0\ne\boldsymbol\Sigma_1$, so its boundary is a small closed curve around the tight class 0 (accuracy about 0.99).
The model's *assumption*, not its capacity to separate, is what fails.

```python
r = np.random.default_rng(5)
X2 = np.r_[r.normal(0, np.sqrt(0.1), (200, 2)), r.normal([4, 0], 3, (200, 2))]
y2 = np.r_[np.zeros(200, int), np.ones(200, int)]
acc_lda_ex2 = LinearDiscriminantAnalysis().fit(X2, y2).score(X2, y2)
acc_qda_ex2 = QuadraticDiscriminantAnalysis().fit(X2, y2).score(X2, y2)
print(acc_lda_ex2, acc_qda_ex2)
```

</details>

### Exercise 19 · A feature map for concentric rings
*💻 Code · ★★☆*

`make_circles` gives an inner and an outer ring. Fit `LogisticRegression` on the raw 2-D inputs and on $\boldsymbol\phi(\mathbf{x}) = (x_1^2, x_2^2)$.
Store both training accuracies. What does the learned boundary $w_1x_1^2 + w_2x_2^2 + b = 0$ look like in the original space?

In [ ]:
X_c, y_c = make_circles(n_samples=400, factor=0.35, noise=0.06, random_state=0)
acc_raw = None
acc_phi = None

if acc_raw is not None:
    check('raw ~ chance (< 0.7)', acc_raw < 0.7, True)
    check('phi ~ perfect (> 0.97)', acc_phi > 0.97, True)
else:
    check('raw ~ chance (< 0.7)', None, True)

<details>
<summary><b>💡 Hint</b></summary>

`np.c_[X_c[:, 0]**2, X_c[:, 1]**2]`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

No line separates two concentric rings, so the raw model is near chance. In $\boldsymbol\phi$-space the classes are separated by the line
$z_1 + z_2 \approx r^2$, which is an **ellipse/circle** $w_1x_1^2 + w_2x_2^2 = -b$ in the input space: a linear model on non-linear features.

```python
X_c, y_c = make_circles(n_samples=400, factor=0.35, noise=0.06, random_state=0)
acc_raw = LogisticRegression().fit(X_c, y_c).score(X_c, y_c)
Phi_c = np.c_[X_c[:, 0] ** 2, X_c[:, 1] ** 2]
m_phi = LogisticRegression().fit(Phi_c, y_c)
acc_phi = m_phi.score(Phi_c, y_c)
print(acc_raw, acc_phi, m_phi.coef_, m_phi.intercept_)
```

</details>

### Exercise 20 · Reconstruct w from the dual and test the support-vector property
*💻 Code · ★★★*

Fit `SVC(kernel='linear', C=1e4)` on the separable data below. scikit-learn stores $\lambda_ny_n$ in `dual_coef_` for the support vectors.

1. Rebuild $\mathbf{w} = \sum_{n\in SV}\lambda_ny_n\mathbf{x}_n$ (`w_dual`) and check it equals `coef_`.
2. Verify $\sum_n\lambda_ny_n = 0$ (`sum_lambda_y`).
3. Move one **non**-support vector further away from the boundary, refit, and store the new `coef_` in `w_moved`. Do the same for logistic regression
   and store how much its coefficients changed in `logreg_change`.

In [ ]:
r = np.random.default_rng(7)
X_sv = np.r_[r.normal([-2, -2], 0.8, (30, 2)), r.normal([2, 2], 0.8, (30, 2))]
y_sv = np.r_[-np.ones(30), np.ones(30)]
svm = SVC(kernel='linear', C=1e4).fit(X_sv, y_sv)

w_dual = None
sum_lambda_y = None
w_moved = None
logreg_change = None

check('w from dual = coef_', w_dual, svm.coef_[0], tol=1e-6)
check('sum lambda_n y_n = 0', sum_lambda_y, 0.0, tol=1e-6)
check('SVM unchanged after moving a non-SV', w_moved, svm.coef_[0], tol=1e-3)
check('logistic regression does change (> 1e-3)', None if logreg_change is None else logreg_change > 1e-3, True)

<details>
<summary><b>💡 Hint</b></summary>

`svm.support_` gives the indices of the SVs; pick a point with $|f(\mathbf{x})| > 1.2$ (clearly outside the margin) but not too far away, and move it by `+3*y*[1,1]`. A point that is already very far has $\sigma_n\approx y_n$ and barely affects logistic regression either.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$\mathbf{w}$ is a sparse combination of the few support vectors, and $\sum\lambda_ny_n = 0$ is the $\partial L/\partial b$ condition. Moving a point that stays outside
the margin leaves $\lambda_n = 0$ and the hyperplane unchanged, whereas every point contributes to the logistic-regression gradient, so its weights move.

```python
r = np.random.default_rng(7)
X_sv = np.r_[r.normal([-2, -2], 0.8, (30, 2)), r.normal([2, 2], 0.8, (30, 2))]
y_sv = np.r_[-np.ones(30), np.ones(30)]
svm = SVC(kernel='linear', C=1e4).fit(X_sv, y_sv)

w_dual = svm.dual_coef_[0] @ svm.support_vectors_
sum_lambda_y = svm.dual_coef_[0].sum()
f_abs = np.abs(svm.decision_function(X_sv))
cand = np.where(f_abs > 1.2)[0]                     # clearly outside the margin: lambda_n = 0
i = cand[np.argmin(f_abs[cand])]                    # the closest such point (still matters for logreg)
X_mv = X_sv.copy(); X_mv[i] += 3 * y_sv[i] * np.array([1, 1])
w_moved = SVC(kernel='linear', C=1e4).fit(X_mv, y_sv).coef_[0]
lr_a = LogisticRegression().fit(X_sv, y_sv).coef_[0]
lr_b = LogisticRegression().fit(X_mv, y_sv).coef_[0]
logreg_change = np.abs(lr_a - lr_b).max()
print('SVs:', len(svm.support_), ' logreg change:', logreg_change.round(4))
```

</details>

---
*Done? Revisit the exercises you needed hints for in a few days — retrieval beats rereading. Back to [Lecture Notes 1-12](../Lecture%20Notes%201-12.md).*